In [1]:
import pandas as pd 

df = pd.read_parquet("../data/processed/arxiv_ml.parquet")

In [3]:
print(df.shape)



(1000, 6)


In [5]:
display(df.head())

,id,title,authors,category,text_raw,text_ml
0,704.0001,Calculation of prompt diphoton production cros...,"C. Bal\'azs, E. L. Berger, P. M. Nadolsky, C.-...",hep-ph,Calculation of prompt diphoton production cros...,calculation prompt diphoton production cross s...
1,704.0002,Sparsity-certifying Graph Decompositions,Ileana Streinu and Louis Theran,math.CO,Sparsity-certifying Graph Decompositions We ...,sparsity certify graph decomposition describe ...
2,704.0003,The evolution of the Earth-Moon system based o...,Hongjun Pan,physics.gen-ph,The evolution of the Earth-Moon system based o...,evolution earth moon system base dark matter f...
3,704.0004,A determinant of Stirling cycle numbers counts...,David Callan,math.CO,A determinant of Stirling cycle numbers counts...,determinant stirling cycle number count unlabe...
4,704.0005,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...,Wael Abu-Shammala and Alberto Torchinsky,math.CA,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...,dyadic lambda alpha lambda alpha compute lambd...


In [6]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
print(df.isnull().sum())

print(df["title"].duplicated().sum())

id          0
title       0
authors     0
category    0
text_raw    0
text_ml     0
dtype: int64
0


In [8]:
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    min_df=2
)

tfidf_matrix = tfidf.fit_transform(df["text_ml"])

print(tfidf_matrix.shape)

(1000, 5000)


In [9]:
cosine_sim = cosine_similarity(tfidf_matrix)

print(cosine_sim.shape)

(1000, 1000)


In [10]:
cosine_sim[:5, :5]

array([[1.        , 0.00252264, 0.00392936, 0.        , 0.00549176],
       [0.00252264, 1.        , 0.01108079, 0.00798555, 0.01311308],
       [0.00392936, 0.01108079, 1.        , 0.        , 0.        ],
       [0.        , 0.00798555, 0.        , 1.        , 0.        ],
       [0.00549176, 0.01311308, 0.        , 0.        , 1.        ]])

In [11]:
print(tfidf_matrix.shape)

print(cosine_sim.shape)

print(cosine_sim[:5, :5])

(1000, 5000)
(1000, 1000)
[[1.         0.00252264 0.00392936 0.         0.00549176]
 [0.00252264 1.         0.01108079 0.00798555 0.01311308]
 [0.00392936 0.01108079 1.         0.         0.        ]
 [0.         0.00798555 0.         1.         0.        ]
 [0.00549176 0.01311308 0.         0.         1.        ]]


In [12]:
indices = pd.Series(df.index, index=df["title"]).drop_duplicates()

indices.head()

title
Calculation of prompt diphoton production cross sections at Tevatron and\n  LHC energies      0
Sparsity-certifying Graph Decompositions                                                      1
The evolution of the Earth-Moon system based on the dark matter field\n  fluid model          2
A determinant of Stirling cycle numbers counts unlabeled acyclic\n  single-source automata    3
From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\alpha}$                                          4
dtype: int64

In [13]:
def recommend_papers(title, top_n=5):

    # Find paper index
    idx = indices[title]

    # Similarity scores
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort by similarity
    sim_scores = sorted(
        sim_scores,
        key=lambda x: x[1],
        reverse=True
    )

    # Remove the paper itself
    sim_scores = sim_scores[1:top_n+1]

    # Extract indices
    paper_indices = [i[0] for i in sim_scores]

    # Get recommendations
    recommendations = df.iloc[
        paper_indices
    ][
        [
            "title",
            "authors",
            "category"
        ]
    ].copy()

    recommendations["Similarity"] = [
        score[1]
        for score in sim_scores
    ]

    return recommendations

In [14]:
df.iloc[0]["title"]

'Calculation of prompt diphoton production cross sections at Tevatron and\n  LHC energies'

In [15]:
recommend_papers(
    df.iloc[0]["title"],
    top_n=5
)

,title,authors,category,Similarity
263,Gluon Radiation of an Expanding Color Skyrmion...,Jian Dai,hep-ph,0.330879
839,Associated production of the charged Higgs bos...,"Yao-Bei Liu, Jie-Fen Shen",hep-ph,0.317494
105,Multiple Parton Scattering in Nuclei: Quark-qu...,"Andreas Schafer, Xin-Nian Wang and Ben-Wei Zhang",hep-ph,0.294623
293,QED x QCD Resummation and Shower/ME Matching f...,B.F.L. Ward and S.A. Yost,hep-ph,0.273019
253,Unravelling the sbottom spin at the CERN LHC,Alexandre Alves and Oscar Eboli,hep-ph,0.220093


In [ ]:
def recommend_papers(title, top_n=5):

    # Find paper index
    idx = indices[title]

    # Similarity scores
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort by similarity
    sim_scores = sorted(
        sim_scores,
        key=lambda x: x[1],
        reverse=True
    )

    # Remove the paper itself
    sim_scores = sim_scores[1:top_n+1]

    # Extract indices
    paper_indices = [i[0] for i in sim_scores]

    # Get recommendations
    recommendations = df.iloc[
        paper_indices
    ][[
        "title",
        "authors",
        "category",
        "text_raw"
    ]].copy()

    recommendations["Similarity (%)"] = [
        round(score[1] * 100, 2)
        for score in sim_scores
    ]


    return recommendations